In [ ]:
# 07_explain_breakdown.ipynb
# Post-hoc Explainability using BreakDown

import sys
import time
import psutil
import os
import tracemalloc
sys.path.append("../src")
from utils import DATASETS, TARGET, RANDOM_STATE, N_INSTANCES


import pandas as pd
import numpy as np
import joblib
import dalex as dx
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit

processed_path = Path("../datasets/processed")
models_path = Path("../results/black_box_models")
results_path = Path("../results/Breakdown")
results_path.mkdir(parents=True, exist_ok=True)

for ds in DATASETS:
    print(f"\n=== Dataset: {ds} ===")

    data = pd.read_csv(processed_path / f"{ds}_processed.csv")
    X = data.drop(columns=[TARGET])
    y = data[TARGET]

    s = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
    train_idx, test_idx = next(s.split(X, y))
    X_train = X.iloc[train_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    X_test  = X.iloc[test_idx].reset_index(drop=True)
    y_test  = y.iloc[test_idx].reset_index(drop=True)

    defect_indices = np.where(y_test.values == 1)[0]
    if len(defect_indices) == 0:
        print(f"  No defective instances in {ds}")
        continue

    ds_model_path = models_path / ds
    ds_result_path = results_path / ds
    ds_result_path.mkdir(parents=True, exist_ok=True)

    for model_dir in sorted(ds_model_path.iterdir()):
        if not model_dir.is_dir():
            continue

        model_name = model_dir.name
        print(f"Model: {model_name}")

        model = joblib.load(model_dir / "model.joblib")
        model_result_path = ds_result_path / model_name
        model_result_path.mkdir(parents=True, exist_ok=True)

        explainer = dx.Explainer(
            model=model,
            data=X_train,
            y=y_train,
            label=model_name,
            verbose=False
        )

        records = []
        bd_times = []

        _mem_before_loop = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
        tracemalloc.start()

        for i, idx in enumerate(defect_indices[:N_INSTANCES]):
            instance = X_test.iloc[idx:idx + 1]

            _t0 = time.perf_counter()
            breakdown = explainer.predict_parts(instance, type="break_down")
            bd_times.append(time.perf_counter() - _t0)

            if breakdown is None or breakdown.result is None:
                continue

            df_bd = pd.DataFrame(breakdown.result)

            if "contribution" not in df_bd.columns:
                continue

            df_bd = df_bd[~df_bd["variable"].isin(["_baseline_", "_full_model_"])]
            contribs = df_bd["contribution"].astype(float).values

            model_pred = int(model.predict(instance)[0])
            prob_defect = float(model.predict_proba(instance)[0, 1])

            plot_obj = breakdown.plot(show=False)
            if plot_obj is not None:
                plot_obj.write_html(
                    str(model_result_path / f"instance_{i}_breakdown.html")
                )
        
        _, _peak_traced = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        _mem_after_loop = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
        _peak_mb = max(_mem_after_loop - _mem_before_loop, _peak_traced / (1024 * 1024))

        if bd_times:
            timing_entry = {
                "Dataset": ds, "Model": model_name, "Method": "Breakdown",
                "Explain_Time_Mean_s": round(float(sum(bd_times)/len(bd_times)), 4),
                "Explain_Time_Std_s":  round(float(pd.Series(bd_times).std()), 4),
                "Mem_Delta_MB": round(_peak_mb, 2)
            }
            timing_csv = model_result_path / "breakdown_timing.csv"
            pd.DataFrame([timing_entry]).to_csv(timing_csv, index=False)
            print(f"   Breakdown time: {timing_entry['Explain_Time_Mean_s']:.4f}s ± {timing_entry['Explain_Time_Std_s']:.4f}s | Peak Mem: {timing_entry['Mem_Delta_MB']:.1f} MB")


=== Dataset: CM1 ===
Model: Gradient_Boosting
   Breakdown time: 0.6099s ± 0.1709s | Peak Mem: 57.0 MB
Model: MLP
   Breakdown time: 0.6468s ± 0.1339s | Peak Mem: 28.8 MB
Model: Random_Forest
   Breakdown time: 7.3680s ± 0.1938s | Peak Mem: 29.0 MB
Model: SVM
   Breakdown time: 2.4115s ± 0.8389s | Peak Mem: 28.7 MB

=== Dataset: JM1 ===
Model: Gradient_Boosting
   Breakdown time: 2.1631s ± 1.1230s | Peak Mem: 45.5 MB
Model: MLP
   Breakdown time: 1.0761s ± 0.1720s | Peak Mem: 28.7 MB
Model: Random_Forest
   Breakdown time: 12.0183s ± 1.0884s | Peak Mem: 119.1 MB
Model: SVM
